In [1]:
# debug_sphere.ipynb
import numpy as np
import sys, os
sys.path.insert(0, os.getcwd())

from src.geometry import Sphere, SphericalCap
from src.geometry import sphere_sphere_surface_distance
from src.truncation import split_sphere_by_box
from src.connectivity import build_connectivity, particle_surface_distance

L = 10000.0
r = 200.0
thresh = 1.8
half = L/2

print("="*70)
print("第一步：验证 split_sphere_by_box 对跨越边界球的切割")
print("="*70)

s = Sphere(np.array([4900., 0., 0.]), r, id=0)
parts = split_sphere_by_box(s, L)

print(f"原始球: center={s.c}, r={s.r}")
print(f"切割后片段数: {len(parts)}")
for i, p in enumerate(parts):
    print(f"  片段{i}: {type(p).__name__}")
    if hasattr(p, 'c'):
        print(f"    center={p.c}")
    if hasattr(p, 'n') and hasattr(p, 'd'):
        print(f"    n={p.n}, d={p.d}")

print("\n" + "="*70)
print("第二步：验证粒子间距离计算（用分割后的片段）")
print("="*70)

if len(parts) >= 2:
    # 同一原始球的两个片段之间的距离
    dist_12 = particle_surface_distance(parts[0], parts[1])
    print(f"片段0 与 片段1 的粒子间距离: {dist_12:.4f} nm")
    print(f"  阈值: {thresh} nm, 是否连通: {dist_12 <= thresh}")
    print("  物理上同一原始球的两个部分相距一个周期，不应因位置接近而连通")

print("\n" + "="*70)
print("第三步：验证距离计算函数是否会被某个 '默认保守估计' 分支阻断")
print("="*70)

# 测试各种组合
s1 = Sphere(np.array([0., 0., 0.]), r, id=1)
s2 = Sphere(np.array([400., 0., 0.]), r, id=2)
# 这两个球表面距离为0，应连通
dist_ss = particle_surface_distance(s1, s2)
print(f"两个完整球 (中心距400): 距离={dist_ss:.4f}, 是否连通={dist_ss <= thresh}")

# 球冠与完整球
if len(parts) >= 1:
    dist_cap_sphere = particle_surface_distance(parts[0], s1)
    print(f"球冠与完整球: 距离={dist_cap_sphere:.4f}, 是否连通={dist_cap_sphere <= thresh}")

# 两个球冠
if len(parts) >= 2:
    dist_cap_cap = particle_surface_distance(parts[0], parts[1])
    print(f"两个球冠: 距离={dist_cap_cap:.4f}, 是否连通={dist_cap_cap <= thresh}")

print("\n" + "="*70)
print("第四步：用一排球（从左到右）测试 build_connectivity 是否正常工作")
print("="*70)

# 生成从左到右的一排球，确保接触电极
n_spheres = 35
x_positions = np.linspace(-4800, 4800, n_spheres)
spheres_chain = []
for i, x in enumerate(x_positions):
    spheres_chain.append(Sphere(np.array([x, 0., 0.]), r, id=i))

# 分割（完全在盒内，应该每个球返回1个片段）
segments_chain = []
for s in spheres_chain:
    segments_chain.extend(split_sphere_by_box(s, L))

print(f"生成 {len(spheres_chain)} 个球，分割后 {len(segments_chain)} 个片段")
print(f"  预期: {len(spheres_chain)} 个片段")

conn, uf = build_connectivity(segments_chain, L, thresh)
print(f"build_connectivity 结果: {conn}")

if not conn:
    print("\n🔍 调试: 检查电极接触和根节点")
    LEFT_NODE = ("PLANE", "LEFT")
    RIGHT_NODE = ("PLANE", "RIGHT")
    left_root = uf.find(LEFT_NODE)
    right_root = uf.find(RIGHT_NODE)
    print(f"左电极根: {left_root}")
    print(f"右电极根: {right_root}")
    print(f"两者是否相同: {left_root == right_root}")
    
    # 检查每个片段归属
    for i, seg in enumerate(segments_chain[:5]):  # 只打印前5个
        print(f"  片段{i}: 根={uf.find(i)}, center={seg.c}")

print("\n" + "="*70)
print("第五步：测试电极接触判定是否被意外过滤")
print("="*70)

# 单个球接触左电极
s_left = Sphere(np.array([-4800., 0., 0.]), r, id=99)
seg_left = split_sphere_by_box(s_left, L)
print(f"左电极球: center={s_left.c}, r={s_left.r}")
print(f"  左表面 = {s_left.c[0] - s_left.r} = {-4800-200} = {-5000}")
print(f"  是否接触左电极: 是")

conn_left, uf_left = build_connectivity(seg_left, L, thresh)
LEFT_NODE = ("PLANE", "LEFT")
print(f"单个左电极球: build_connectivity 结果 = {conn_left}")
print(f"  预期: False (因为没有右电极连接)")

# 检查该球是否真的连接到了左电极
left_root = uf_left.find(LEFT_NODE)
print(f"  左电极根: {left_root}")
for i, seg in enumerate(seg_left):
    print(f"  片段{i}: 根={uf_left.find(i)}, center={seg.c}")
    if uf_left.find(i) == left_root:
        print(f"    ✅ 片段{i} 连接到左电极")

第一步：验证 split_sphere_by_box 对跨越边界球的切割
原始球: center=[4900.    0.    0.], r=200.0
切割后片段数: 2
  片段0: SphericalCap
    center=[4900.    0.    0.]
    n=[-1.  0.  0.], d=-100.0
  片段1: SphericalCap
    center=[-5100.     0.     0.]
    n=[1. 0. 0.], d=100.0

第二步：验证粒子间距离计算（用分割后的片段）
片段0 与 片段1 的粒子间距离: 9600.0000 nm
  阈值: 1.8 nm, 是否连通: False
  物理上同一原始球的两个部分相距一个周期，不应因位置接近而连通

第三步：验证距离计算函数是否会被某个 '默认保守估计' 分支阻断
两个完整球 (中心距400): 距离=0.0000, 是否连通=True
球冠与完整球: 距离=4500.0000, 是否连通=False
两个球冠: 距离=9600.0000, 是否连通=False

第四步：用一排球（从左到右）测试 build_connectivity 是否正常工作
生成 35 个球，分割后 35 个片段
  预期: 35 个片段
build_connectivity 结果: True

第五步：测试电极接触判定是否被意外过滤
左电极球: center=[-4800.     0.     0.], r=200.0
  左表面 = -5000.0 = -5000 = -5000
  是否接触左电极: 是
单个左电极球: build_connectivity 结果 = False
  预期: False (因为没有右电极连接)
  左电极根: 0
  片段0: 根=0, center=[-4800.     0.     0.]
    ✅ 片段0 连接到左电极


In [1]:
import numpy as np
from src.geometry import Sphere
from src.truncation import split_sphere_by_box
from src.connectivity import build_connectivity
from src.monte_carlo import sample_random_spheres

L = 10000.0
r = 200.0
thresh = 1.8

# ============================================================
# 测试1：人为紧密排列的球（确定导通）
# ============================================================
print("="*70)
print("测试1：人为紧密排列的球（球心距=300nm，确保连通）")
print("="*70)

# 沿 X 轴排列，间距 300nm，覆盖从 -4800 到 4800
x_positions = np.arange(-4800, 4801, 300)  # 33个点
spheres = [Sphere(np.array([x, 0., 0.]), r, id=i) for i, x in enumerate(x_positions)]

segments = []
for s in spheres:
    segments.extend(split_sphere_by_box(s, L))

print(f"球数: {len(spheres)}, 片段数: {len(segments)}")
connected, uf = build_connectivity(segments, L, thresh)
print(f"导通: {connected}")
print()

# ============================================================
# 测试2：随机分布 500 个球（体积分数约 1.67%）
# ============================================================
print("="*70)
print("测试2：随机分布 500 个球（f ≈ 1.67%）")
print("="*70)

N = 500
rng = np.random.default_rng(42)
centers = rng.uniform(-L/2, L/2, size=(N, 3))
spheres_rand = [Sphere(centers[i], r, id=i) for i in range(N)]

segments_rand = []
for s in spheres_rand:
    segs = split_sphere_by_box(s, L)
    segments_rand.extend(segs)

print(f"球数: {N}, 片段数: {len(segments_rand)}")
connected, uf = build_connectivity(segments_rand, L, thresh)
print(f"导通: {connected}")
print()

# ============================================================
# 测试3：随机分布 2000 个球（体积分数约 6.7%）
# ============================================================
print("="*70)
print("测试3：随机分布 2000 个球（f ≈ 6.7%）")
print("="*70)

N2 = 2000
rng2 = np.random.default_rng(123)
centers2 = rng2.uniform(-L/2, L/2, size=(N2, 3))
spheres_rand2 = [Sphere(centers2[i], r, id=i) for i in range(N2)]

segments_rand2 = []
for s in spheres_rand2:
    segs = split_sphere_by_box(s, L)
    segments_rand2.extend(segs)

print(f"球数: {N2}, 片段数: {len(segments_rand2)}")
connected2, uf2 = build_connectivity(segments_rand2, L, thresh)
print(f"导通: {connected2}")
print()

# ============================================================
# 测试4：随机分布 5000 个球（体积分数约 16.7%）
# ============================================================
print("="*70)
print("测试4：随机分布 5000 个球（f ≈ 16.7%）")
print("="*70)

N3 = 5000
rng3 = np.random.default_rng(456)
centers3 = rng3.uniform(-L/2, L/2, size=(N3, 3))
spheres_rand3 = [Sphere(centers3[i], r, id=i) for i in range(N3)]

segments_rand3 = []
for s in spheres_rand3:
    segs = split_sphere_by_box(s, L)
    segments_rand3.extend(segs)

print(f"球数: {N3}, 片段数: {len(segments_rand3)}")
connected3, uf3 = build_connectivity(segments_rand3, L, thresh)
print(f"导通: {connected3}")

测试1：人为紧密排列的球（球心距=300nm，确保连通）
球数: 33, 片段数: 33
导通: True

测试2：随机分布 500 个球（f ≈ 1.67%）
球数: 500, 片段数: 565
导通: False

测试3：随机分布 2000 个球（f ≈ 6.7%）
球数: 2000, 片段数: 2245
导通: False

测试4：随机分布 5000 个球（f ≈ 16.7%）
球数: 5000, 片段数: 5639
导通: False


In [4]:
N_high = 6000  # f ≈ 20%
spheres_high = sample_random_spheres(N_high, L, r, 42)  # 去掉 seed=，直接用位置参数
segments_high = []
for s in spheres_high:
    segments_high.extend(split_sphere_by_box(s, L))
connected_high, _ = build_connectivity(segments_high, L, thresh)
print(f"f≈20%, N={N_high}, 导通: {connected_high}")

f≈20%, N=6000, 导通: False
